# PDD / Exit Dose / Exit Fluence Comparison Workflow

This notebook is organized in **clean blocks** so you can:
1. Load simulation outputs (`pdd`, `exitDose`, `exitFluence`) from text files.
2. Compare simulated PDD against your measured curves for each applicator and mode.
3. Compare exit-dose and exit-fluence in practical, physics-based ways.
4. Estimate source-parameter changes to improve simulation-vs-measurement agreement.

## 1) Imports and plot style

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['legend.fontsize'] = 10


## 2) Measured PDD library (your provided data)

Use key format: `(applicator_cm, mode)` where mode is one of: `flash9`, `conv9`, `flash6alt`.

In [ ]:
MEASURED_PDD = {
    (10, 'flash9'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
            21,22,23,24,25,26,27,28,29,30,31,33,35,37,39,41,43,45,47,49,
            51,56,61,66
        ], dtype=float),
        'norm_pct': np.array([
            94.97,94.97,95.15,97.66,98.18,98.66,99.09,99.35,99.52,99.57,
            99.78,99.91,99.96,100.00,99.91,99.78,99.52,99.31,98.87,98.79,
            97.83,97.40,96.62,95.58,94.54,93.33,91.68,90.21,88.04,85.83,
            84.06,81.46,73.92,65.87,57.63,48.27,38.99,30.20,21.92,15.08,
            9.10,5.42,0.52,0.43
        ], dtype=float),
    },
    (10, 'conv9'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
            21,22,23,24,25,26,27,28,29,30,31,33,35,37,39,41,43,45,47,49,
            51,56,61,66
        ], dtype=float),
        'norm_pct': np.array([
            94.62,94.39,96.64,96.86,97.31,98.65,97.76,98.88,99.10,99.10,
            98.65,99.10,99.10,100.00,98.65,99.33,98.43,98.21,97.31,97.53,
            96.19,96.19,94.39,94.39,92.60,91.03,90.36,87.89,85.87,83.18,
            80.27,74.22,66.82,58.97,51.57,40.81,32.06,23.54,15.92,9.64,
            5.38,1.12,1.12,1.12
        ], dtype=float),
    },
    (10, 'flash6alt'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
            21,22,23,24,25,26,27,28,29,30,31,33,35,37,39,41,43,45,47,49,
            51,56,61,66
        ], dtype=float),
        'norm_pct': np.array([
            92.59,94.51,95.91,97.13,98.15,98.72,99.36,99.84,100.00,99.96,
            99.57,99.04,98.02,96.42,94.32,92.08,88.89,85.44,81.48,76.76,
            71.52,66.28,60.34,54.79,48.72,43.04,37.04,31.35,25.93,20.75,
            16.09,8.49,3.83,1.53,0.64,0.38,0.38,0.38,0.38,0.38,
            0.38,0.38,0.38,0.38
        ], dtype=float),
    },
    (5, 'flash9'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
            21,22,23,24,25,26,27,28,29,30,31,33,35,37,39,41,43,45,47,49,
            51,56,61,66
        ], dtype=float),
        'norm_pct': np.array([
            93.22,93.74,94.99,95.84,96.41,97.04,97.61,97.84,98.52,98.80,
            99.32,99.43,99.69,99.83,99.83,100.00,99.91,99.60,99.43,98.97,
            98.46,97.72,96.53,95.44,94.08,92.71,90.66,88.78,86.39,83.66,
            80.64,73.80,66.29,57.63,48.92,39.61,31.18,22.98,15.72,9.97,
            5.52,0.80,0.28,0.28
        ], dtype=float),
    },
    (5, 'conv9'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
            21,22,23,24,25,26,27,28,29,30,31,33,35,37,39,41,43,45,47,49,
            51,56,61,66
        ], dtype=float),
        'norm_pct': np.array([
            93.95,94.39,95.88,96.03,96.92,97.37,97.22,98.86,97.82,99.01,
            98.86,99.16,99.26,99.65,99.90,100.00,99.53,99.16,99.16,99.01,
            98.56,97.37,96.03,95.73,93.80,92.16,89.63,87.54,85.16,82.48,
            78.46,71.02,63.42,54.49,45.41,36.03,27.69,19.65,13.25,8.04,
            4.47,0.89,0.60,0.60
        ], dtype=float),
    },
    (5, 'flash6alt'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
            21,22,23,24,25,26,27,28,29,30,31,33,35,37,39,41,43,45,47,49,
            51,56,61,66
        ], dtype=float),
        'norm_pct': np.array([
            89.45,90.70,92.55,94.30,95.64,97.07,98.16,98.83,99.41,99.92,
            100.00,99.75,99.16,98.24,96.31,94.14,91.04,88.02,83.63,78.98,
            73.83,68.17,62.14,56.28,50.00,44.22,38.27,32.29,26.63,21.36,
            16.83,9.05,4.27,1.68,0.59,0.25,0.17,0.17,0.17,0.08,
            0.08,0.08,0.08,0.08
        ], dtype=float),
    },
    (2, 'flash9'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,17,19,21,23,25,27,29,31,35,39,43,47,51,61
        ], dtype=float),
        'norm_pct': np.array([
            96.81,97.33,97.96,98.48,98.80,99.32,99.63,100.00,99.79,99.69,
            99.32,98.59,97.80,96.49,94.82,89.69,82.88,75.34,67.07,59.37,
            51.88,44.45,37.80,25.89,16.10,8.19,2.98,0.68,0.13
        ], dtype=float),
    },
    (2, 'conv9'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,17,19,21,23,25,27,29,31,35,39,43,47,51,61
        ], dtype=float),
        'norm_pct': np.array([
            96.81,97.33,97.96,98.48,98.80,99.32,99.63,100.00,99.79,99.69,
            99.32,98.59,97.80,96.49,94.82,89.69,82.88,75.34,67.07,59.37,
            51.88,44.45,37.80,25.89,16.10,8.19,2.98,0.68,0.13
        ], dtype=float),
    },
    (2, 'flash6alt'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,17,19,21,23,25,27,29,31,35,39,43,47,51,61
        ], dtype=float),
        'norm_pct': np.array([
            96.81,97.33,97.96,98.48,98.80,99.32,99.63,100.00,99.79,99.69,
            99.32,98.59,97.80,96.49,94.82,89.69,82.88,75.34,67.07,59.37,
            51.88,44.45,37.80,25.89,16.10,8.19,2.98,0.68,0.13
        ], dtype=float),
    },
}

sorted(MEASURED_PDD.keys())


## 3) File readers (pdd / exitDose / exitFluence)

Supported formats:
- Simple 2-column: `[depth_or_position_mm, value]`
- Geant4 mesh scorer export: `iX, iY, iZ, total(value), total(val^2), entry`

For mesh scorer files, the notebook automatically:
- Uses `iZ` for PDD depth and `total(value)` as dose.
- Detects the varying index axis for exit profiles (`iX`/`iY`/`iZ`) and uses `total(value)` as profile value.
- Converts index to mm with configurable bin size/offset controls in the user block.

In [ ]:
def _safe_read_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')

    df = pd.read_csv(
        path,
        sep=r'[\s,;]+',
        engine='python',
        comment='#',
        header=None,
    )
    df = df.dropna(axis=1, how='all')
    if df.empty:
        raise ValueError(f'No numeric rows found in {path}')
    return df


def _looks_like_mesh_scorer(df):
    return df.shape[1] >= 4 and df.shape[1] <= 6


def _pick_profile_axis(df):
    axes = {0: df.iloc[:, 0].nunique(), 1: df.iloc[:, 1].nunique(), 2: df.iloc[:, 2].nunique()}
    best_axis = max(axes, key=axes.get)
    return best_axis


def read_pdd_file(path, depth_bin_mm=1.0, depth_offset_mm=0.0):
    df = _safe_read_table(path)

    if df.shape[1] == 2:
        out = pd.DataFrame({'depth_mm': df.iloc[:, 0].astype(float), 'dose': df.iloc[:, 1].astype(float)})
        return out.sort_values('depth_mm').drop_duplicates('depth_mm').reset_index(drop=True)

    if _looks_like_mesh_scorer(df):
        iZ = df.iloc[:, 2].astype(float)
        dose = df.iloc[:, 3].astype(float)
        out = pd.DataFrame({'depth_idx': iZ, 'dose': dose})
        out = out.groupby('depth_idx', as_index=False)['dose'].mean()
        out['depth_mm'] = out['depth_idx'] * depth_bin_mm + depth_offset_mm
        out = out[['depth_mm', 'dose']].sort_values('depth_mm').reset_index(drop=True)
        return out

    raise ValueError('Unrecognized PDD format. Expected 2 columns or mesh scorer columns (iX,iY,iZ,value,...).')


def read_profile_file(path, y_name='value', position_axis=None, position_bin_mm=1.0, position_offset_mm=0.0):
    df = _safe_read_table(path)

    if df.shape[1] == 2:
        out = pd.DataFrame({'position_mm': df.iloc[:, 0].astype(float), y_name: df.iloc[:, 1].astype(float)})
        return out.sort_values('position_mm').drop_duplicates('position_mm').reset_index(drop=True)

    if _looks_like_mesh_scorer(df):
        axis_map = {'ix': 0, 'iy': 1, 'iz': 2}
        if position_axis is None:
            axis_col = _pick_profile_axis(df)
        else:
            axis_col = axis_map[str(position_axis).lower()]

        pos_idx = df.iloc[:, axis_col].astype(float)
        val = df.iloc[:, 3].astype(float)
        out = pd.DataFrame({'pos_idx': pos_idx, y_name: val})
        out = out.groupby('pos_idx', as_index=False)[y_name].mean()
        out['position_mm'] = out['pos_idx'] * position_bin_mm + position_offset_mm
        out = out[['position_mm', y_name]].sort_values('position_mm').reset_index(drop=True)
        return out

    raise ValueError(f'Unrecognized profile format in {path}. Expected 2 columns or mesh scorer columns.')


## 4) Choose scenario and load simulation files

Update paths if needed.

In [ ]:
# ---- USER CONTROLS ----
applicator_cm = 10         # 10, 5, or 2
mode = 'flash9'            # 'flash9', 'conv9', 'flash6alt'

pdd_file = 'build/pdd_1MJ50026028_T1.txt'
exit_dose_file = 'build/exitDose_1M_J50026028_T1.txt'
exit_fluence_file = 'build/exitFluence_1M_J50026028_T1.txt'

# Mesh-index to mm conversion (used when files are Geant4 scorer exports)
pdd_depth_bin_mm = 1.0
pdd_depth_offset_mm = 0.0
exit_position_bin_mm = 1.0
exit_position_offset_mm = 0.0

# Optional: force profile axis for mesh files: 'ix', 'iy', 'iz', or None for auto
exit_profile_axis = None
# -----------------------

meas = MEASURED_PDD[(applicator_cm, mode)]
meas_df = pd.DataFrame({'depth_mm': meas['depth_mm'], 'meas_norm_pct': meas['norm_pct']})

sim_pdd_df = read_pdd_file(
    pdd_file,
    depth_bin_mm=pdd_depth_bin_mm,
    depth_offset_mm=pdd_depth_offset_mm,
)
sim_exit_dose_df = read_profile_file(
    exit_dose_file,
    y_name='exit_dose',
    position_axis=exit_profile_axis,
    position_bin_mm=exit_position_bin_mm,
    position_offset_mm=exit_position_offset_mm,
)
sim_exit_fluence_df = read_profile_file(
    exit_fluence_file,
    y_name='exit_fluence',
    position_axis=exit_profile_axis,
    position_bin_mm=exit_position_bin_mm,
    position_offset_mm=exit_position_offset_mm,
)

print('Loaded:')
print(f'  measured points      : {len(meas_df)}')
print(f'  simulated PDD        : {len(sim_pdd_df)}')
print(f'  simulated exitDose   : {len(sim_exit_dose_df)}')
print(f'  simulated exitFluence: {len(sim_exit_fluence_df)}')

print('\nPDD preview:')
display(sim_pdd_df.head(10))
print('\nExit dose preview:')
display(sim_exit_dose_df.head(10))
print('\nExit fluence preview:')
display(sim_exit_fluence_df.head(10))


## 5) Literature-style PDD normalization + alignment

Typical comparison practice for electron PDD curves:
- Normalize each curve to **100% at dmax**.
- Interpolate simulation onto measured depths.
- Compare shape metrics (R90/R80/R50, practical range) and pointwise errors.

In [ ]:
def normalize_to_dmax(depth_mm, y):
    i = np.argmax(y)
    dmax = depth_mm[i]
    ymax = y[i]
    y_norm = 100.0 * y / ymax if ymax != 0 else np.zeros_like(y)
    return y_norm, dmax


def interp_at(x_src, y_src, x_tgt):
    return np.interp(x_tgt, x_src, y_src)


def find_depth_at_percent(depth_mm, pdd_norm, pct):
    i_max = int(np.argmax(pdd_norm))
    x = depth_mm[i_max:]
    y = pdd_norm[i_max:]
    if len(x) < 2:
        return np.nan

    for i in range(len(y)-1):
        y1, y2 = y[i], y[i+1]
        if (y1 >= pct >= y2) or (y1 <= pct <= y2):
            x1, x2 = x[i], x[i+1]
            if y2 == y1:
                return float(x1)
            t = (pct - y1) / (y2 - y1)
            return float(x1 + t*(x2 - x1))
    return np.nan


meas_norm, meas_dmax = normalize_to_dmax(meas_df['depth_mm'].values, meas_df['meas_norm_pct'].values)
sim_norm, sim_dmax = normalize_to_dmax(sim_pdd_df['depth_mm'].values, sim_pdd_df['dose'].values)

sim_on_meas = interp_at(sim_pdd_df['depth_mm'].values, sim_norm, meas_df['depth_mm'].values)

comp_df = pd.DataFrame({
    'depth_mm': meas_df['depth_mm'].values,
    'measured_norm_pct': meas_norm,
    'sim_norm_pct_interp': sim_on_meas,
})
comp_df['delta_pctpts'] = comp_df['sim_norm_pct_interp'] - comp_df['measured_norm_pct']
comp_df['abs_delta_pctpts'] = comp_df['delta_pctpts'].abs()

metrics = {
    'mode': mode,
    'applicator_cm': applicator_cm,
    'meas_dmax_mm': float(meas_dmax),
    'sim_dmax_mm': float(sim_dmax),
    'dmax_shift_mm(sim-meas)': float(sim_dmax - meas_dmax),
    'MAE_pctpts': float(comp_df['abs_delta_pctpts'].mean()),
    'RMSE_pctpts': float(np.sqrt(np.mean(comp_df['delta_pctpts']**2))),
    'max_abs_diff_pctpts': float(comp_df['abs_delta_pctpts'].max()),
    'meas_R90_mm': float(find_depth_at_percent(meas_df['depth_mm'].values, meas_norm, 90)),
    'sim_R90_mm': float(find_depth_at_percent(sim_pdd_df['depth_mm'].values, sim_norm, 90)),
    'meas_R50_mm': float(find_depth_at_percent(meas_df['depth_mm'].values, meas_norm, 50)),
    'sim_R50_mm': float(find_depth_at_percent(sim_pdd_df['depth_mm'].values, sim_norm, 50)),
}
metrics


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(meas_df['depth_mm'], meas_norm, 'o-', label='Measured (norm to dmax)')
plt.plot(sim_pdd_df['depth_mm'], sim_norm, '-', lw=2, label='Simulation (norm to dmax)')
plt.xlabel('Depth (mm)')
plt.ylabel('PDD (%)')
plt.title(f'PDD comparison | {applicator_cm} cm | {mode}')
plt.ylim(-2, 105)
plt.legend()
plt.show()

plt.figure(figsize=(8,4))
plt.axhline(0, color='k', lw=1)
plt.plot(comp_df['depth_mm'], comp_df['delta_pctpts'], 'o-')
plt.xlabel('Depth (mm)')
plt.ylabel('Sim - Meas (percentage points)')
plt.title('Pointwise residuals (interpolated)')
plt.show()

pd.DataFrame([metrics])


## 6) Exit-dose and exit-fluence comparison block

What to compare with:
- **Exit dose profile**: compare against downstream detector dose profile / film / chamber scan at same plane.
- **Exit fluence profile**: compare against fluence-sensitive measurement (e.g., film/scintillator response after correction) or use as a shape diagnostic.

If you have measured exit profiles, place them in files with two columns: `position_mm value`, then set paths below.

In [ ]:
# Optional measured profile files (set to None if not available)
measured_exit_dose_file = None
measured_exit_fluence_file = None


def normalize_profile(y):
    m = np.max(y)
    return 100.0 * y / m if m != 0 else np.zeros_like(y)


def profile_metrics(x_ref, y_ref, x_cmp, y_cmp, label='profile'):
    y_cmp_on_ref = np.interp(x_ref, x_cmp, y_cmp)
    diff = y_cmp_on_ref - y_ref
    return {
        f'{label}_MAE_pctpts': float(np.mean(np.abs(diff))),
        f'{label}_RMSE_pctpts': float(np.sqrt(np.mean(diff**2))),
        f'{label}_max_abs_pctpts': float(np.max(np.abs(diff))),
    }

sim_exit_dose_norm = normalize_profile(sim_exit_dose_df['exit_dose'].values)
sim_exit_fluence_norm = normalize_profile(sim_exit_fluence_df['exit_fluence'].values)

fig, ax = plt.subplots(1, 2, figsize=(13,4))
ax[0].plot(sim_exit_dose_df['position_mm'], sim_exit_dose_norm, label='Sim exit dose')
ax[0].set_title('Exit dose profile (normalized)')
ax[0].set_xlabel('Position (mm)')
ax[0].set_ylabel('Relative (%)')
ax[0].set_ylim(-2, 105)
ax[0].legend()

ax[1].plot(sim_exit_fluence_df['position_mm'], sim_exit_fluence_norm, label='Sim exit fluence', color='tab:orange')
ax[1].set_title('Exit fluence profile (normalized)')
ax[1].set_xlabel('Position (mm)')
ax[1].set_ylabel('Relative (%)')
ax[1].set_ylim(-2, 105)
ax[1].legend()

plt.tight_layout()
plt.show()

summary = {}
if measured_exit_dose_file:
    meas_exit_dose_df = read_profile_file(measured_exit_dose_file, y_name='meas_exit_dose')
    meas_y = normalize_profile(meas_exit_dose_df['meas_exit_dose'].values)
    sim_y = sim_exit_dose_norm
    plt.figure(figsize=(8,4))
    plt.plot(meas_exit_dose_df['position_mm'], meas_y, 'o-', label='Measured exit dose')
    plt.plot(sim_exit_dose_df['position_mm'], sim_y, '-', label='Sim exit dose')
    plt.title('Exit dose comparison')
    plt.xlabel('Position (mm)')
    plt.ylabel('Relative (%)')
    plt.legend()
    plt.show()
    summary.update(profile_metrics(meas_exit_dose_df['position_mm'].values, meas_y,
                                   sim_exit_dose_df['position_mm'].values, sim_y,
                                   label='exit_dose'))

if measured_exit_fluence_file:
    meas_exit_flu_df = read_profile_file(measured_exit_fluence_file, y_name='meas_exit_fluence')
    meas_y = normalize_profile(meas_exit_flu_df['meas_exit_fluence'].values)
    sim_y = sim_exit_fluence_norm
    plt.figure(figsize=(8,4))
    plt.plot(meas_exit_flu_df['position_mm'], meas_y, 'o-', label='Measured exit fluence')
    plt.plot(sim_exit_fluence_df['position_mm'], sim_y, '-', label='Sim exit fluence')
    plt.title('Exit fluence comparison')
    plt.xlabel('Position (mm)')
    plt.ylabel('Relative (%)')
    plt.legend()
    plt.show()
    summary.update(profile_metrics(meas_exit_flu_df['position_mm'].values, meas_y,
                                   sim_exit_fluence_df['position_mm'].values, sim_y,
                                   label='exit_fluence'))

if summary:
    display(pd.DataFrame([summary]))
else:
    print('No measured exit profiles provided yet. Added plotting + metrics hooks.')


## 7) Parameter-tuning guidance block (how to make simulated PDD fit measured)

This block gives **data-driven hints** on which beam/source parameters are likely off.

In [ ]:
def suggest_source_tuning(metrics_dict):
    suggestions = []
    dmax_shift = metrics_dict['dmax_shift_mm(sim-meas)']
    sim_r50 = metrics_dict['sim_R50_mm']
    meas_r50 = metrics_dict['meas_R50_mm']

    if np.isfinite(dmax_shift):
        if dmax_shift > 1.0:
            suggestions.append('Simulated dmax is too deep -> decrease mean beam energy slightly.')
        elif dmax_shift < -1.0:
            suggestions.append('Simulated dmax is too shallow -> increase mean beam energy slightly.')

    if np.isfinite(sim_r50) and np.isfinite(meas_r50):
        dr50 = sim_r50 - meas_r50
        if dr50 > 1.5:
            suggestions.append('R50 too deep in simulation -> reduce mean energy or add low-energy tail filtering.')
        elif dr50 < -1.5:
            suggestions.append('R50 too shallow in simulation -> increase mean energy or reduce energy spread.')

    mae = metrics_dict['MAE_pctpts']
    if mae > 4:
        suggestions.append('Overall shape mismatch is large -> tune source angular spread and spot size (scatter conditions).')

    surface_sim = comp_df.loc[comp_df['depth_mm'].idxmin(), 'sim_norm_pct_interp']
    surface_meas = comp_df.loc[comp_df['depth_mm'].idxmin(), 'measured_norm_pct']
    if surface_sim - surface_meas > 2:
        suggestions.append('Surface dose too high in simulation -> narrow angular spread or refine upstream material model.')
    elif surface_sim - surface_meas < -2:
        suggestions.append('Surface dose too low in simulation -> slightly broaden angular spread / include additional scatter.')

    if not suggestions:
        suggestions.append('Fit is already reasonably close. Next: fine-scan mean energy in small steps and minimize RMSE.')

    return suggestions


tuning_suggestions = suggest_source_tuning(metrics)
print('Suggested beam/source adjustments:')
for i, s in enumerate(tuning_suggestions, 1):
    print(f'{i}. {s}')


## 8) Optional mini grid-search template (offline workflow)

Use this table to track reruns when you change source parameters in Geant4 macro files.

In [ ]:
run_log = pd.DataFrame(columns=[
    'run_id', 'mean_energy_MeV', 'energy_sigma_MeV', 'spot_sigma_mm', 'ang_sigma_deg',
    'dmax_shift_mm', 'R50_shift_mm', 'RMSE_pctpts', 'MAE_pctpts'
])
run_log
